<img src="https://backend.burla.dev/static/welcome.svg" width="45%">

#### This notebook prices an option with **200 million Monte Carlo paths** on Burla, in about 3 minutes. <br/> It's only 4 steps — follow along!

### What is Burla?

Burla is the simplest way to run Python across hundreds or thousands of cloud machines.
One function — `remote_parallel_map` — fans your code out across isolated Docker containers, grows the cluster on demand, and streams results back. Missing packages on the workers? Burla auto-installs them.

No Dockerfiles, no cluster YAML, no queue. Just Python.  
Burla is free to try — this whole notebook runs on machines we spin up for you.

### What will this demo do?

We'll price a European call option by simulating **200 million asset-price paths** under geometric Brownian motion. On one laptop this is a minute or two of pure NumPy.

On Burla, we split the 200M paths into **200 independent chunks** and run them all at the same time on 200 cloud workers. Each worker returns a tiny dict (`sum`, `sum_sq`, `n`) — no huge arrays fly across the network — and the client aggregates into a final price estimate and a Monte Carlo standard error.

At the end you'll see the price, the stderr, and a histogram of per-chunk estimates collapsing onto the true value.

## 1)&nbsp; Boot some machines (200+ CPUs):
&nbsp;&nbsp;&nbsp;&nbsp;Sign in using the button below, then hit the **`⏻ Start`** button on your dashboard homepage.  
&nbsp;&nbsp;&nbsp;&nbsp;This will boot a small cluster in Google Cloud for you. Burla is free to try so this is on us!

&nbsp;&nbsp;
<a href="https://login.burla.dev">
    <img src="https://i.ibb.co/QFNncHcp/Clean-Shot-2026-03-19-at-18-13-07.jpg" width="28%">
</a>

## 2)&nbsp; Install the Python package:

In [ ]:
!uv pip install burla

## 3)&nbsp; Authorize this computer:

In [ ]:
!burla login

## 4)&nbsp; Run 200 million Monte Carlo paths on Burla 🚀

Each chunk seeds its own RNG (`42 + chunk_id`) so the 200 streams are reproducible and statistically independent. Burla spins up 200 workers, runs `run_chunk` on each, and returns the list of results.

The first call takes a few seconds longer while Burla installs `numpy` into each container — subsequent runs start instantly.

In [ ]:
import math
import numpy as np
from burla import remote_parallel_map

# 200 chunks × 1,000,000 paths = 200,000,000 Monte Carlo paths
N_CHUNKS = 200
PER_CHUNK = 1_000_000

# European call option: spot=100, strike=95, 1 year, 1% rate, 30% volatility
params = {"S0": 100.0, "K": 95.0, "T": 1.0, "r": 0.01, "sigma": 0.3}
tasks = [(i, PER_CHUNK, params) for i in range(N_CHUNKS)]


def run_chunk(chunk_id: int, n: int, p: dict) -> dict:
    import numpy as np

    rng = np.random.default_rng(seed=42 + chunk_id)
    Z = rng.standard_normal(n)
    ST = p["S0"] * np.exp((p["r"] - 0.5 * p["sigma"] ** 2) * p["T"] + p["sigma"] * np.sqrt(p["T"]) * Z)
    payoff = np.maximum(ST - p["K"], 0.0) * np.exp(-p["r"] * p["T"])

    return {
        "chunk_id": chunk_id,
        "n": n,
        "sum": float(payoff.sum()),
        "sum_sq": float((payoff ** 2).sum()),
    }


# Burla grows the cluster on demand and runs the 200 chunks in parallel
results = remote_parallel_map(run_chunk, tasks, func_cpu=1, func_ram=2, grow=True)

total_n = sum(r["n"] for r in results)
total_sum = sum(r["sum"] for r in results)
total_sum_sq = sum(r["sum_sq"] for r in results)

mean = total_sum / total_n
stderr = math.sqrt(((total_sum_sq / total_n) - mean ** 2) / total_n)

print(f"price estimate: {mean:.6f}   stderr: {stderr:.6f}   (n = {total_n:,})")

### What just happened?

You just ran **200 million Monte Carlo paths** across 200 cloud containers in a few seconds. On one laptop core this is 1–2 minutes of NumPy; the same code at full scale (1 billion paths, 2,000 chunks — see `main.py`) finishes in around a minute on Burla.

Notice there's no `asyncio`, no `Pool`, no Spark — the worker function is a normal pure-Python function that takes a tuple and returns a dict. Burla handles everything else.

### Plot the chunks

200 independent chunks means 200 independent estimates of the same price. Plotting them as a histogram shows the Monte Carlo error collapsing around the true value.

In [ ]:
import matplotlib.pyplot as plt

chunk_means = [r["sum"] / r["n"] for r in results]

plt.figure(figsize=(10, 4))
plt.hist(chunk_means, bins=40)
plt.axvline(mean, color="red", linestyle="--", label=f"overall = {mean:.4f}")
plt.xlabel("per-chunk mean payoff")
plt.ylabel("# chunks")
plt.title(f"{N_CHUNKS:,} independent chunks × {PER_CHUNK:,} paths each")
plt.legend()
plt.show()

### Try this next

- Change `N_CHUNKS` to `2000` and `PER_CHUNK` to `500_000` → 1 billion paths.
- Tweak `params["sigma"]` up to `0.6` and re-run — the stderr will widen, the histogram fattens.
- Swap `grow=True` for `grow=False` if your cluster is already sized for the job.
- Replace `run_chunk` with your own simulator (VaR, Bayesian posterior, physics sim). Same structure, same scale-out.

## Thank you for trying Burla! ❤️

### Run the full 1-billion-path version

See [`main.py`](https://github.com/Burla-Cloud/monte-carlo-simulation/blob/main/main.py) in this repo — 2,000 chunks × 500,000 paths each, same code.

### Run this on your laptop 🧑‍💻

1. `pip install burla`
2. `burla login`
3. Run the same example code from above ☝️

Congrats! Your laptop now effectively has 1000+ CPUs, and 4000G of RAM &nbsp; : )

<br/>

### Any way we can help? Lets chat!

Schedule a meeting: https://cal.com/jakez  
Send me an email: jake@burla.dev